<a href="https://colab.research.google.com/github/AshishGit22/impcode/blob/main/stock_researcher_multi_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## stock researcher tool - based on technical analysis
## consists of 3 agents - market, momentum and risk
## and one final agent combining the outputs of all 3 and giving the final decision
## llm model : gemini-2.5-flash

In [ ]:
# Disclaimer:
# This analysis is generated for educational and research purposes only.
# It does not constitute investment advice, financial advice, or a recommendation to buy or sell any security.
# Market investments involve risk, and past performance is not indicative of future results.
# Please consult a qualified financial advisor before making any investment decisions.

In [6]:
!pip install -q yfinance pandas numpy ta google-generativeai

  Preparing metadata (setup.py) ... done


In [7]:
from google.colab import userdata
import google.generativeai as genai

import yfinance as yf
import pandas as pd
import numpy as np
import ta


In [8]:
api_key = userdata.get('stock_researcher_api')

if api_key is None:
    raise ValueError("Gemini API key not found in Colab secrets")

genai.configure(api_key=api_key)

model = genai.GenerativeModel(
    model_name="gemini-2.5-flash"
)


In [9]:
## model check

prompt = "give me poem about nature in 8 lines"

response = model.generate_content(prompt)

print(response.text)


The emerald world, a vibrant scene,
Where gentle breezes whisper keen.
Sunlight paints the forest floor,
And birds sing tunes, then ask for more.
Wildflowers bloom in colors bright,
Reflecting nature's pure delight.
A quiet stream, a calming sound,
Peace in this beauty can be found.


In [10]:
## llm calling function

def call_gemini(prompt):
  response = model.generate_content(prompt)
  return response.text.strip()

In [11]:
## fetch Indian stock data

def fetch_stock(symbol, period='6mo'):

  df = yf.download(symbol+'.NS', period=period)
  df.columns = df.columns.get_level_values(0)
  df.reset_index(inplace=True)
  return df

In [33]:
## adding indicators

def add_indicators(df):
  close = df['Close'].astype(float)
  df['SMA_20'] = ta.trend.sma_indicator(close, window=20)
  df['RSI'] = ta.momentum.rsi(close, window=14)
  return df

In [14]:
## building context for agents

In [16]:
def build_market_context(df):
  latest = df.iloc[-1]
  return {"close": float(latest["Close"]),
          "sma_20": float(latest["SMA_20"])}

In [17]:
def build_momentum_context(df):
  latest = df.iloc[-1]
  return {"rsi_14": float(latest["RSI"])}

In [18]:
def build_risk_context(df):
  latest = df.iloc[-1]
  return {"price_vs_sma": float(latest["Close"] - latest["SMA_20"]),
          "rsi_14": float(latest["RSI"])}

In [19]:
## building agents

In [22]:
def market_agent(context):
  prompt = f"""
You are a Stock Market Trend Agent.

Data:
- Close Price: {context['close']:.2f}
- 20-day SMA: {context['sma_20']:.2f}

Return:
Trend: <Uptrend | Downtrend | Sideways>
Strength: <Weak | Moderate | Strong>
Explanation: <1 sentence>
"""
  return call_gemini(prompt)

In [23]:
def momentum_agent(context):
  prompt = f"""
You are a Stock Momentum Agent.

Data:
- RSI (14): {context['rsi_14']:.2f}

Return:
Momentum State: <Overbought | Oversold | Neutral>
Momentum Bias: <Bullish | Bearish | Neutral>
Explanation: <1 sentence>
"""
  return call_gemini(prompt)

In [24]:
def risk_agent(context):
  prompt = f"""
You are a Stock Market - Risk Assessment Agent.

Data:
- Price minus SMA: {context['price_vs_sma']:.2f}
- RSI: {context['rsi_14']:.2f}

Return:
Risk Level: <Low | Medium | High>
Risks: <2 bullet points>
"""
  return call_gemini(prompt)

In [26]:
## decision agent - combining and inferring the output of all 3 agents

def decision_agent(market_out, momentum_out, risk_out):
  prompt = f"""
You are the Final Decision Agent.

You are NOT designing rules or logic.
You are applying judgment based on expert opinions.

Return ONLY the final verdict.

Format:
Decision: <Buy | Hold | Sell>
Confidence: <Low | Medium | High>
Explanation: <2–3 sentences>

Market Agent Assessment:
{market_out}

Momentum Agent Assessment:
{momentum_out}

Risk Agent Assessment:
{risk_out}

Constraints:
- No price targets
- No guaranteed returns
- Neutral professional tone
"""
  return call_gemini(prompt)


In [42]:
symbol = 'SBIN'

df = fetch_stock(symbol)

df = add_indicators(df)

# df.tail(5)

market_ctx = build_market_context(df)
momentum_ctx = build_momentum_context(df)
risk_ctx = build_risk_context(df)
# market_ctx
# momentum_ctx
# risk_ctx

market_output = market_agent(market_ctx)
momentum_output = momentum_agent(momentum_ctx)
risk_output = risk_agent(risk_ctx)
# market_output
# momentum_output
# risk_output


final_decision = decision_agent(market_output, momentum_output, risk_output)
final_decision

/tmp/ipython-input-2541511667.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol+'.NS', period=period)
[*********************100%***********************]  1 of 1 completed


'Decision: Hold\nConfidence: Medium\nExplanation: The asset demonstrates strong bullish momentum within an established uptrend. However, it is currently in significantly overbought territory and extended above its moving average, presenting a high risk of an imminent price pullback or consolidation. Therefore, a Hold decision is prudent, advising caution for new entries and monitoring for existing positions.'

In [43]:
print(final_decision)

Decision: Hold
Confidence: Medium
Explanation: The asset demonstrates strong bullish momentum within an established uptrend. However, it is currently in significantly overbought territory and extended above its moving average, presenting a high risk of an imminent price pullback or consolidation. Therefore, a Hold decision is prudent, advising caution for new entries and monitoring for existing positions.


In [35]:
df.tail(5)

Price,Date,Close,High,Low,Open,Volume,SMA_20,RSI
121,2026-01-23,1029.500000,1053.000000,1025.599976,1051.900024,10917698,1012.920004,59.632314
122,2026-01-27,1053.150024,1055.000000,1030.250000,1034.099976,14420630,1017.325006,66.761567
123,2026-01-28,1063.500000,1065.599976,1044.150024,1059.400024,12171257,1021.827505,69.315566
124,2026-01-29,1066.199951,1075.000000,1060.300049,1063.000000,14800724,1026.027502,69.963938
125,2026-01-30,1077.150024,1082.500000,1060.400024,1064.000000,9480769,1030.647504,72.501738
